# SCF Abstraction: Testing the kernel
In this notebook we will test the 4c scf kernel, in order to determine if it works for $Ne$, $He$, and $H^-$.

In [1]:
import numpy as np
from py_mods.src.SCF_4c_dev.types_4c import CS_4c_KU_SCF_Context
from py_mods.src.external.DIRAC_ME import (
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_h5,
    build_uncontracted_basis_from_h5,
)

from py_mods.src.SCF_4c_dev.KUSCF_dev import (
    occupation_4c,
    eri_classified,
    _kuscf_kernel,
)

# 1. $He$ Test

The expected energy after convergence is:

$$
E_{He} = -2.861285099218
$$

In [2]:
def scf_state_allocation_from_h5(h5_filename, tot_charge=0):
    S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
    H_nuc = V + W + T
    nuc_charge = get_nuc_charge(h5_filename)
    n_elec = nuc_charge - (tot_charge)

    _, nL, nS = build_uncontracted_basis_from_h5(h5_filename)
    eri = full_eri_from_h5(h5_filename)
    eri = eri_classified(eri, nL)
  


    occ_det = occupation_4c(nS, nL, n_elec)

    test_ctx = CS_4c_KU_SCF_Context(
        nL,
        nS,
        S,
        T,
        V,
        W,
        eri,
        n_elec,
        theta=0.00,
        occ=occ_det,
        p_guess="core",
        verbose=True,
        threshold=5e-8,
        acc_hist_size=10,
        max_iter=500
    )

    return test_ctx

In [3]:
He_cxt = scf_state_allocation_from_h5("data/He_checkpoint.h5", tot_charge=0)
print(He_cxt.occ)

He_kuscf_results = _kuscf_kernel(He_cxt)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2           -2.7497446361133155+0.0000000000000000j           -2.7497446361133155+0.0000000000000000j     1.3789E+00
    3           -2.8599296846696469+0.0000000000000000j           -0.1101850485563314+0.0000000000000000j     8.5446E-02
    4           -2.8612543623062789-0.0000000000000000j           -0.0013246776366320-0.0000000000000000j    

In [4]:
-1.2059228886845643--1.2059228886845510

-1.3322676295501878e-14

In [5]:
print(He_kuscf_results.e_orb)

[-3.80623188e+04 -3.80623188e+04 -3.76499522e+04 -3.76499522e+04
 -3.75875437e+04 -3.75875437e+04 -3.75875437e+04 -3.75875437e+04
 -3.75807164e+04 -3.75807164e+04 -3.75663781e+04 -3.75663781e+04
 -3.75663781e+04 -3.75663781e+04 -3.75639694e+04 -3.75639694e+04
 -3.75631869e+04 -3.75631869e+04 -3.75611467e+04 -3.75611467e+04
 -3.75611467e+04 -3.75611467e+04 -3.75602534e+04 -3.75602534e+04
 -3.75602534e+04 -3.75602534e+04 -3.75593196e+04 -3.75593196e+04
 -3.75582932e+04 -3.75582932e+04 -3.75582932e+04 -3.75582932e+04
 -3.75580117e+04 -3.75580117e+04 -3.75579662e+04 -3.75579662e+04
 -3.75579662e+04 -3.75579662e+04 -3.75579662e+04 -3.75579662e+04
 -3.75579266e+04 -3.75579266e+04 -3.75578013e+04 -3.75578013e+04
 -3.75578013e+04 -3.75578013e+04 -3.75577333e+04 -3.75577333e+04
 -3.75577333e+04 -3.75577333e+04 -9.17658935e-01 -9.17658935e-01
  6.17824483e-01  6.17824483e-01  2.51775731e+00  2.51775731e+00
  2.51789502e+00  2.51789502e+00  2.51789502e+00  2.51789502e+00
  3.60924381e+00  3.60924

In [6]:
assert np.abs(He_kuscf_results.E_SCF - -2.861285099218) < 1e-10

# 2. $H$ Test

the reference energy for $H^-$ is:

$$
E_{H^-} =  -0.4999521765048605
$$



In [7]:
H_cxt = scf_state_allocation_from_h5("data/H_checkpoint.h5", tot_charge=0)
H_kuscf_results = _kuscf_kernel(H_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2           -0.4999521765048612+0.0000000000000000j           -0.4999521765048612+0.0000000000000000j     3.6043E-11
Convergence achieved after 2 iterations.


In [8]:
assert np.abs(H_kuscf_results.E_SCF - -0.4999521765048605) < 1e-10

# 3. $Ne$ Test

The $Ne$ refrerence energy is:

$$
E_{Ne} = -128.68579576537914
$$

In [9]:
Ne_cxt = scf_state_allocation_from_h5("data/Ne_checkpoint.h5", tot_charge=0)

In [10]:
print(Ne_cxt.occ)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [11]:
Ne_kuscf_results = _kuscf_kernel(Ne_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2         -112.4356540807836780-0.0000000000000003j         -112.4356540807836780-0.0000000000000003j     3.6087E+01
    3         -119.6726220088576724-0.0000000000000006j           -7.2369679280739945-0.0000000000000003j     1.0547E+01
    4         -125.9578217479771496-0.0000000000000000j           -6.2851997391194772+0.0000000000000005j     1.1530E+01
    5         -127.5649059806221430-0.0000000000000001j           -1.6070842326449934-0.0000000000000000j     5.7383E+00
    6 

In [12]:
assert np.abs(Ne_kuscf_results.E_SCF -  -128.68579576537914) < 1e-10

In [13]:
Ne_kuscf_results.e_orb

array([-6.93656331e+04, -6.93656331e+04, -4.58001019e+04, -4.58001019e+04,
       -3.99651967e+04, -3.99651967e+04, -3.92832920e+04, -3.92832920e+04,
       -3.92832920e+04, -3.92832920e+04, -3.83455467e+04, -3.83455467e+04,
       -3.81467162e+04, -3.81467162e+04, -3.81467162e+04, -3.81467162e+04,
       -3.80643533e+04, -3.80643533e+04, -3.79425564e+04, -3.79425564e+04,
       -3.79425564e+04, -3.79425564e+04, -3.78410638e+04, -3.78410638e+04,
       -3.78039859e+04, -3.78039859e+04, -3.78039859e+04, -3.78039859e+04,
       -3.76931478e+04, -3.76931478e+04, -3.76694745e+04, -3.76694745e+04,
       -3.76694745e+04, -3.76694745e+04, -3.76658180e+04, -3.76658180e+04,
       -3.76601026e+04, -3.76601026e+04, -3.76601026e+04, -3.76601026e+04,
       -3.76210071e+04, -3.76210071e+04, -3.76208878e+04, -3.76208878e+04,
       -3.76208878e+04, -3.76208878e+04, -3.76208878e+04, -3.76208878e+04,
       -3.76100442e+04, -3.76100442e+04, -3.76100442e+04, -3.76100442e+04,
       -3.76023687e+04, -